[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lmassaron/finetuning/blob/main/chapter_09/listing_9.2.ipynb)

In [ ]:
import sys
if "google.colab" in sys.modules:
    !pip install -q -U gptqmodel transformers


### Listing 9.2: Quantizing a merged model to 4-bit using AutoGPTQ

In [1]:
from gptqmodel import GPTQModel, QuantizeConfig
from transformers import AutoTokenizer
import torch

quantize_config = QuantizeConfig(
    bits=4,
    group_size=128,
    desc_act=False
)

model = GPTQModel.load(
    "./merged-model",
    quantize_config=quantize_config,
    torch_dtype=torch.float16,
    device="cuda:0"
)

tokenizer = AutoTokenizer.from_pretrained("./merged-model")
examples = [
    {k: v for k, v in tokenizer(text, return_tensors="pt", truncation=True, max_length=512).items()}
    for text in [
        "Quantization calibration example text",
        "The quick brown fox jumps over the lazy dog",
        "Large language models are trained on diverse datasets",
    ]
]

model.quantize(examples)
model.save_quantized("./merged-model-gptq")

WARN  Python GIL is enabled: Multi-gpu quant acceleration for MoE models is sub-optimal and multi-core accelerated cpu packing is also disabled. We recommend Python >= 3.13.3t with Pytorch > 2.8 for mult-gpu quantization and multi-cpu packing with env `PYTHON_GIL=0`.


INFO  ENV: Auto setting PYTORCH_ALLOC_CONF='expandable_segments:True,max_split_size_mb:256,garbage_collection_threshold:0.7' for memory saving.


INFO  ENV: Auto setting CUDA_DEVICE_ORDER=PCI_BUS_ID for correctness.          


INFO  

┌─────────────┐    ┌────────────────────────┐    ┌────────────┐    ┌─────────┐
│ GPT-QModel  │ -> │ ▓▓▓▓▓▓▓▓▓▓▓▓ 16bit     │ -> │ ▒▒▒▒ 8bit  │ -> │ ░░ 4bit │
└─────────────┘    └────────────────────────┘    └────────────┘    └─────────┘
GPT-QModel   : 7.0.0+fb4058f
Transformers : 5.7.0
Torch        : 2.11.0+cu130
Triton       : 3.6.0


INFO  QuantizeConfig: offload_to_disk_path auto set to temporary dir `/tmp/gptqmodel_xtfeyye2`


...

Files in directory:
model.safetensors
tokenizer_config.json
chat_template.jinja
generation_config.json
config.json
quantize_config.json
quant_log.csv
tokenizer.json
Content of saved `generation_config.json`:
{
    "_from_model_config": true,
    "bos_token_id": 1,
    "do_sample": true,
    "eos_token_id": 2,
    "transformers_version": "5.7.0"
}
Content of saved `config.json`:
{
    "architectures": [
        "MistralForCausalLM"
    ],
    "attention_dropout": 0.0,
    "bos_token_id": 1,
    "dtype": "float16",
    "eos_token_id": 2,
    "head_dim": 128,
    "hidden_act": "silu",
    "hidden_size": 4096,
    "initializer_range": 0.02,
    "intermediate_size": 14336,
    "max_position_embeddings": 32768,
    "model_type": "mistral",
    "num_attention_heads": 32,
    "num_hidden_layers": 32,
    "num_key_value_heads": 8,
    "pad_token_id": 0,
    "quantization_config": {
        "bits": 4,
        "checkpoint_format": "gptq",
        "desc_act": false,
        "format": "gptq",
     

INFO  Module: Total direct tensors materialized from lazy checkpoint source: 3 


INFO  Pre-Quantized model size: 13812.54MB, 13.49GB                            


INFO  Quantized model size: 3963.36MB, 3.87GB                                  


INFO  Size difference: 9849.18MB, 9.62GB - 71.31%                              
